In [ ]:
#pip install pypdf langchain langchain_openai langchain_community kiwipiepy rank_bm25

In [1]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv("../.env")

# 환경 변수에서 API 키 가져오기
#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = ("Data/투자설명서.pdf")
loader = PyPDFLoader(file_path)

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap = 200)

docs = loader.load_and_split(doc_splitter)

In [4]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

kiwi_tokenizer = Kiwi()

def kiwi_tokenize(text):
  return [token.form for token in kiwi_tokenizer.tokenize(text)] #type:ignore

In [5]:
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 2

In [6]:
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI

# 관련있는 문서를 수집 후, Chatgpt로 최종 답변까지 수행하는 체인을 생성
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0.2, model="gpt-4o"),
    chain_type="stuff",
    retriever=bm25_retriever,
    return_source_documents=True # 답변에 사용된 source document도 보여주도록 설정
)


In [7]:
qa_chain.invoke("이 회사가 발행한 주식의 총 발행량이 어느정도야?")

{'query': '이 회사가 발행한 주식의 총 발행량이 어느정도야?',
 'result': '이 회사가 발행한 주식의 총 발행량은 13,602,977주입니다.',
 'source_documents': [Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': 'Data/투자설명서.pdf', 'total_pages': 514, 'page': 301, 'page_label': '302'}, page_content='실패하더라도 성실 수행을 입증할 수 있는 경우에는 제제 조치를 면제 받을 수 있으나,\n불성실 수행으로 연구개발의 결과가 극히 불량하여 최종 평가 시 실패 과제로 결정된\n경우에는 정부로부터 받은 지원금의 전부 혹은 일부의 환수 또는 향후 정부출연과제\n참여 제한의 제제 조치가 가해질 수 있습니다.\n▶동사는 특수관계자와의 거래시 이사회 운영 규정 제10조에 따라 사전에 이사회 결\n의를 거쳐야 하며, 내부회계 관리 규정 등 동사 내부통제기준에 따라 거래 투명성을 확\n보하는 등 절차를 준수하고 있습니다. 향후에도 특수관계자와의 거래가 발생하게 될\n경우 특수관계자에 대한 부당한 이익 제공 등에 해당하지 않는 범위에서 거래가 진행\n되도록 사전에 내부 규정에 따른 관련 절차를 준수할 예정입니다. 만일 특수관계자와\n의 거래 조건 등이 제3자와의 거래와 비교하여 적정성이 유지되지 않을 경우 이는 주\n주의 이익을 침해하는 결과를 초래할 수 있습니다.\n▶실권주를 인수한 대표주관회사 또는 인수회사가 수익 확정을 위해 인수한 물량을 조\n기에 장내에서 대량 매도할 경우 일시적 물량 출회에 따른 주가 하락의 가능성이 존재\n하며, 각 사에서 인수한 실권주를 